In [1]:
import pandas as pd
import numpy as np


def project_block_rates(
    start_date,
    num_days,
    block_tats,
    initial_block_rates,
    input_rate_schedule,
    freq="D"
):
    """
    Projects block-level movement rates forward using cumulative TAT.

    Parameters
    ----------
    start_date : str
        First date of projection, e.g. "2026-07-01".

    num_days : int
        Number of days to project.

    block_tats : dict
        TAT in days from each block to the next block.
        Example:
        {
            "Block 1": 8.5,
            "Block 2": 11,
            "Block 3": 7,
            "Block 4": 9,
            "Block 5": 6,
            "Block 6": 10
        }

    initial_block_rates : dict
        Initial movement rate for each block before the new input rate reaches it.
        Example:
        {
            "Block 1": 1000,
            "Block 2": 950,
            "Block 3": 900,
            ...
        }

    input_rate_schedule : list of tuples
        List of input rate changes.
        Example:
        [
            ("2026-07-01", 1000),
            ("2026-07-15", 1300)
        ]

    Returns
    -------
    daily_rates : pd.DataFrame
        Daily projected target rate by block.

    weekly_rates : pd.DataFrame
        Weekly average projected target rate by block.
    """

    blocks = list(initial_block_rates.keys())

    # Create projection calendar
    dates = pd.date_range(start=start_date, periods=num_days, freq=freq)
    daily = pd.DataFrame(index=dates)

    # Build input rate timeline
    input_schedule = pd.DataFrame(
        input_rate_schedule,
        columns=["date", "input_rate"]
    )
    input_schedule["date"] = pd.to_datetime(input_schedule["date"])
    input_schedule = input_schedule.sort_values("date")

    daily["input_rate"] = np.nan

    for change_date, rate in zip(input_schedule["date"], input_schedule["input_rate"]):
        daily.loc[daily.index >= change_date, "input_rate"] = rate

    # Fill initial input rate before first change
    daily["input_rate"] = daily["input_rate"].ffill().bfill()

    # Cumulative TAT delay to each block
    cumulative_delay = {}
    delay = 0

    for block in blocks:
        cumulative_delay[block] = delay
        if block in block_tats:
            delay += block_tats[block]

    # Project rate for each block
    for block in blocks:
        lag_days = cumulative_delay[block]

        projected_rates = []

        for current_date in daily.index:
            source_date = current_date - pd.Timedelta(days=lag_days)

            if source_date < daily.index.min():
                # New input has not reached this block yet
                projected_rates.append(initial_block_rates[block])
            else:
                # Use most recent available input rate at or before source date
                prior_dates = daily.index[daily.index <= source_date]

                if len(prior_dates) == 0:
                    projected_rates.append(initial_block_rates[block])
                else:
                    projected_rates.append(daily.loc[prior_dates[-1], "input_rate"])

        daily[block] = projected_rates

    # Add work week
    daily["WW"] = daily.index.to_series().dt.isocalendar().week.astype(int)
    daily["Year"] = daily.index.to_series().dt.isocalendar().year.astype(int)

    weekly_rates = (
        daily
        .groupby(["Year", "WW"])[blocks]
        .mean()
        .reset_index()
    )

    return daily, weekly_rates

In [16]:
block_tats = {
    "Block 1": 8.5,
    "Block 2": 7,
    "Block 3": 7,
    "Block 4": 7,
    "Block 5": 7,
    "Block 6": 14
}

initial_block_rates = {
    "Block 1": 2050,
    "Block 2": 2000,
    "Block 3": 1900,
    "Block 4": 1800,
    "Block 5": 1600,
    "Block 6": 1600,
    "Block 7": 1600
}

input_rate_schedule = [
    ("2026-07-01", 2150),
    ("2026-07-15", 2200)
]

daily_rates, weekly_rates = project_block_rates(
    start_date="2026-07-01",
    num_days=120,
    block_tats=block_tats,
    initial_block_rates=initial_block_rates,
    input_rate_schedule=input_rate_schedule
)

print(weekly_rates)

    Year  WW      Block 1      Block 2      Block 3      Block 4      Block 5  \
0   2026  27  2150.000000  2000.000000  1900.000000  1800.000000  1600.000000   
1   2026  28  2150.000000  2064.285714  1900.000000  1800.000000  1600.000000   
2   2026  29  2185.714286  2150.000000  2007.142857  1800.000000  1600.000000   
3   2026  30  2200.000000  2171.428571  2150.000000  1950.000000  1600.000000   
4   2026  31  2200.000000  2200.000000  2171.428571  2150.000000  1835.714286   
5   2026  32  2200.000000  2200.000000  2200.000000  2171.428571  2150.000000   
6   2026  33  2200.000000  2200.000000  2200.000000  2200.000000  2171.428571   
7   2026  34  2200.000000  2200.000000  2200.000000  2200.000000  2200.000000   
8   2026  35  2200.000000  2200.000000  2200.000000  2200.000000  2200.000000   
9   2026  36  2200.000000  2200.000000  2200.000000  2200.000000  2200.000000   
10  2026  37  2200.000000  2200.000000  2200.000000  2200.000000  2200.000000   
11  2026  38  2200.000000  2

In [22]:
daily_rates

,input_rate,Block 1,Block 2,Block 3,Block 4,Block 5,Block 6,Block 7,WW,Year
2026-07-01,2150.0,2150.0,2000.0,1900.0,1800.0,1600.0,1600.0,1600.0,27,2026
2026-07-02,2150.0,2150.0,2000.0,1900.0,1800.0,1600.0,1600.0,1600.0,27,2026
2026-07-03,2150.0,2150.0,2000.0,1900.0,1800.0,1600.0,1600.0,1600.0,27,2026
2026-07-04,2150.0,2150.0,2000.0,1900.0,1800.0,1600.0,1600.0,1600.0,27,2026
2026-07-05,2150.0,2150.0,2000.0,1900.0,1800.0,1600.0,1600.0,1600.0,27,2026
...,...,...,...,...,...,...,...,...,...,...
2026-10-24,2200.0,2200.0,2200.0,2200.0,2200.0,2200.0,2200.0,2200.0,43,2026
2026-10-25,2200.0,2200.0,2200.0,2200.0,2200.0,2200.0,2200.0,2200.0,43,2026
2026-10-26,2200.0,2200.0,2200.0,2200.0,2200.0,2200.0,2200.0,2200.0,44,2026
2026-10-27,2200.0,2200.0,2200.0,2200.0,2200.0,2200.0,2200.0,2200.0,44,2026
